# **1. Giới thiệu dự án (Introduction)**

## **1.1. Bối cảnh kinh doanh & Vấn đề của công ty Fintech**
Trong bối cảnh nền kinh tế Việt Nam đang phục hồi và tầng lớp trung lưu gia tăng mạnh mẽ, nhu cầu vay tiêu dùng tín chấp qua các nền tảng Fintech kỹ thuật số đang trở thành xu hướng tất yếu. Tuy nhiên, đi kèm với cơ hội mở rộng thị phần là bài toán quản trị rủi ro nợ xấu (Credit Default Risk).

## **1.2. Mục tiêu phân tích**
* Khám phá và phân tích tập dữ liệu 50.000 hồ sơ tín dụng để tìm ra các mẫu hành vi của khách hàng.

* Xây dựng mô hình Học máy (Machine Learning) để định lượng rủi ro vỡ nợ của từng cá nhân.

* Ứng dụng ma trận Thị phần - Tăng trưởng (Growth-Share Matrix) nhằm đề xuất phân khúc khách hàng (Market Segment) mang lại biên độ lợi nhuận cao nhất với rủi ro thấp nhất cho công ty.

# **2. Tiền xử lý dữ liệu (Data Preprocessing & Cleaning)**

## **2.1. Khai báo thư viện (Import Libraries)**

In [1]:
import numpy as np
import scipy.stats as stats
import pandas as pd
np.set_printoptions(precision=4, suppress=True)
pd.options.display.max_columns = 50

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
plt.style.use(['seaborn-v0_8', 'seaborn-v0_8-whitegrid'])
%config InlineBackend.figure_format = 'retina'

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [ ]:
# 1. Cài đặt font chữ hỗ trợ tiếng Việt vào hệ thống Kaggle
!apt-get install -y fonts-dejavu

# 2. Xóa cache font cũ của Matplotlib để cập nhật font mới
import matplotlib as mpl
mpl.font_manager._load_fontmanager(try_read_cache=False)

# 3. Cấu hình Matplotlib sử dụng font mới toàn cục
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans' 

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra
The following NEW packages will be installed:
  fonts-dejavu fonts-dejavu-core fonts-dejavu-extra
0 upgraded, 3 newly installed, 0 to remove and 138 not upgraded.
Need to get 3,085 kB of archives.
After this operation, 10.7 MB of additional disk space will be used.
Ign:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-core all 2.37-2build1
Ign:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-extra all 2.37-2build1
Ign:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-dejavu all 2.37-2build1
Ign:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-core all 2.37-2build1
Ign:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-extra all 2.37-2build1
Ign:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-dejavu all 2.37-2build1

## **2.2. Đọc và kiểm tra tổng quan dữ liệu**

In [ ]:
#1. Đọc file excel
df = "/kaggle/input/datasets/thvann81/data-fba25/FBA 2025 - Customer credit data.xlsx"

# Đọc sheet 1
sheet1 = pd.read_excel(df, sheet_name="Sheet1")
print("=== Sheet1 ===")
print(sheet1)

# Đọc sheet 2
sheet2 = pd.read_excel(df, sheet_name="Sheet2")
print("\n=== Sheet2 ===")
print(sheet2)

## **2.3. Chuẩn hóa dữ liệu**

In [ ]:
print("kiểm tra giá trị thiếu:") 
print(sheet1.isnull().sum())
print("Số dòng trùng lặp:") 
print(sheet1.duplicated().sum())

## **2.4. Khớp dữ liệu (Mapping) theo chuẩn**

In [ ]:
# Gán df bằng dữ liệu từ Sheet1 để thực hiện làm sạch
df = sheet1.copy()

print("--- Bắt đầu làm sạch và chuẩn bị dữ liệu ---")

# 1. Kiểm tra các cột số tiền
print("\n1. Kiểm tra các cột số tiền:")
print(f"Kiểu dữ liệu của 'existing_debt_vnd': {df['existing_debt_vnd'].dtype}")
print(f"Kiểu dữ liệu của 'requested_loan_amount_vnd': {df['requested_loan_amount_vnd'].dtype}")
print(f"Số giá trị thiếu trong 'existing_debt_vnd': {df['existing_debt_vnd'].isnull().sum()}")
print(f"Số giá trị thiếu trong 'requested_loan_amount_vnd': {df['requested_loan_amount_vnd'].isnull().sum()}")

# 2. Chuyển đổi cột 'income_level' sang các nhóm
print("\n2. Chuyển đổi cột 'income_level' sang các nhóm:")

# Do Sheet 1 có đuôi '/tháng' còn Sheet 2 thì không, ta sẽ tạo mapping thủ công dựa trên nội dung Sheet 2 để đảm bảo chính xác
income_level_mapping = {
    'Dưới 8 triệu VND/tháng': 'Thấp (Low)',
    'Từ 8 - 20 triệu VND/tháng': 'Trung bình (Medium)',
    'Từ 20 - 45 triệu VND/tháng': 'Cao (High)',
    'Trên 45 triệu VND/tháng': 'Rất cao (Very High)'
}

# Ánh xạ giá trị
df['income_level_group'] = df['income_level'].map(income_level_mapping)

# Thiết lập thứ tự phân loại
income_level_categories = ['Thấp (Low)', 'Trung bình (Medium)', 'Cao (High)', 'Rất cao (Very High)']
df['income_level_group'] = pd.Categorical(df['income_level_group'], categories=income_level_categories, ordered=True)

print("Giá trị duy nhất và số lượng của 'income_level_group':")
print(df['income_level_group'].value_counts())

# 3. Chuyển đổi cột 'credit_score' sang các nhóm dựa trên Sheet 2.
print("\n3. Chuyển đổi cột 'credit_score' sang các nhóm:")
# Định nghĩa bins: 300-579 (Kém), 580-669 (Trung bình), 670-739 (Tốt), 740-799 (Rất tốt), 800-850 (Xuất sắc)
bins = [299, 579, 669, 739, 799, 850]
labels = ['Kém (Poor)', 'Trung bình (Fair)', 'Tốt (Good)', 'Rất tốt (Very Good)', 'Xuất sắc (Excellent)']

df['credit_score_group'] = pd.cut(df['credit_score'], bins=bins, labels=labels, right=True, include_lowest=True)

print("Giá trị duy nhất và số lượng của 'credit_score_group':")
print(df['credit_score_group'].value_counts())

print("\n--- Hoàn thành làm sạch và chuẩn bị dữ liệu ---")
print(df[['income_level', 'income_level_group', 'credit_score', 'credit_score_group']].head())

In [ ]:
# Loại bỏ khoảng trắng thừa
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Mã hóa nợ xấu (Dành cho BA tính toán tỷ lệ)
df['is_defaulted'] = df['default_status'].map({'Defaulted': 1, 'Paid in full': 0})

# Sử dụng pd.cut để chia nhóm Credit Score (thay cho hàm if-elif)
bins = [300, 580, 670, 740, 800, 850]
labels = ['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
df['credit_rating'] = pd.cut(df['credit_score'], bins=bins, labels=labels, right=False)

Tập dữ liệu ban đầu cho thấy chất lượng thu thập rất tốt: không ghi nhận các giá trị khuyết thiếu **(Null/NaN)** hay dữ liệu trùng lặp **(Duplicates)**. Tuy nhiên, để phục vụ cho việc trực quan hóa và đưa vào mô hình học máy, các biến phân loại (Categorical variables) như **income_level**, **credit_score** đã được chuẩn hóa lại format (loại bỏ khoảng trắng thừa) và ánh xạ (mapping) theo đúng thứ tự logic từ thấp đến cao.

# **3. Phân tích Khám phá Dữ liệu (Exploratory Data Analysis - EDA)**

### **3.1. Phân phối của biến mục tiêu (Khách hàng vỡ nợ - Default Status)**


In [ ]:
# Trực quan hóa các biến số (Numerical Features)
print("\n--- Trực quan hóa các biến số ---")
# Sử dụng các cột số đã được xác định trước đó trong notebook, hoặc tự động chọn
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Loại bỏ các cột ID hoặc cột đã được mã hóa không cần trực quan hóa phân phối số
exclude_cols = ['customer_id', 'income_level_numeric', 'default_status_encoded', 'cluster']
num_cols = [col for col in num_cols if col not in exclude_cols and not col.endswith('_encoded')]

plt.figure(figsize=(18, 15))
for i, col in enumerate(num_cols):
    plt.subplot(round(len(num_cols)/3) + 1, 3, i + 1)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Phân phối của {col}')
    plt.xlabel(col)
    plt.ylabel('Tần số')
    # Áp dụng log scale cho các biến có giá trị lớn và phân bố lệch
    if 'vnd' in col or 'amount' in col or 'debt' in col:
        plt.xscale('log')
plt.tight_layout()
plt.show()

# Trực quan hóa các biến phân loại (Categorical Features)
print("\n--- Trực quan hóa các biến phân loại ---")
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Loại bỏ customer_id không cần thiết cho EDA tổng quan
cat_cols = [col for col in cat_cols if col not in ['customer_id']]

plt.figure(figsize=(18, 15))
for i, col in enumerate(cat_cols):
    plt.subplot(round(len(cat_cols)/2) + 1, 2, i + 1)
    sns.countplot(data=df, y=col, order=df[col].value_counts().index, hue=col, legend=False, palette='viridis')
    plt.title(f'Phân phối của {col}')
    plt.xlabel('Số lượng')
    plt.ylabel(col)
plt.tight_layout()
plt.show()

# 3. Mối quan hệ giữa Default Status và các biến phân loại khác
print("\n--- Mối quan hệ giữa Default Status và các biến phân loại ---")
for col in cat_cols:
    if col != 'default_status':
        plt.figure(figsize=(10, 6))
        sns.countplot(data=df, y=col, hue='default_status', palette='coolwarm', order=df[col].value_counts().index)
        plt.title(f'Phân phối Default Status theo {col}')
        plt.xlabel('Số lượng')
        plt.ylabel(col)
        plt.tight_layout()
        plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='default_status', data=df) # Removed palette as hue is not used
plt.title('Phân phối của Trạng thái Vỡ nợ')
plt.xlabel('Trạng thái Vỡ nợ')
plt.ylabel('Số lượng khách hàng')
plt.show()

# 3.2. Phân tích theo Nhân khẩu học & Việc làm (Độ tuổi, Thu nhập, Tình trạng việc làm)
print("\n### 3.2. Phân tích theo Nhân khẩu học & Việc làm")

# Phân tích Độ tuổi
print("\n#### Phân tích theo Độ tuổi")
plt.figure(figsize=(10, 6))
sns.histplot(df['age'], bins=20, kde=True) # Removed palette as hue is not used
plt.title('Phân phối Độ tuổi Khách hàng')
plt.xlabel('Độ tuổi')
plt.ylabel('Tần suất')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x='default_status', y='age', data=df) # Removed palette as hue is not used
plt.title('Độ tuổi theo Trạng thái Vỡ nợ')
plt.xlabel('Trạng thái Vỡ nợ')
plt.ylabel('Độ tuổi')
plt.show()

# Phân tích Thu nhập
print("\n#### Phân tích theo Thu nhập")
plt.figure(figsize=(10, 6))
sns.countplot(x='income_level_group', hue='default_status', data=df, palette='viridis', order=df['income_level_group'].cat.categories)
plt.title('Phân phối Thu nhập theo Trạng thái Vỡ nợ')
plt.xlabel('Nhóm Thu nhập')
plt.ylabel('Số lượng khách hàng')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Trạng thái Vỡ nợ')
plt.tight_layout()
plt.show()

# Phân tích Tình trạng việc làm
print("\n#### Phân tích theo Tình trạng việc làm")
plt.figure(figsize=(10, 6))
sns.countplot(x='employment_status', hue='default_status', data=df, palette='magma')
plt.title('Phân phối Tình trạng Việc làm theo Trạng thái Vỡ nợ')
plt.xlabel('Tình trạng Việc làm')
plt.ylabel('Số lượng khách hàng')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Trạng thái Vỡ nợ')
plt.tight_layout()
plt.show()
# 3.3. Phân tích theo Hành vi Tín dụng (Điểm tín dụng, Khoản tiết kiệm, Mục đích vay)
print("\n### 3.3. Phân tích theo Hành vi Tín dụng")

# Phân tích Điểm tín dụng
print("\n#### Phân tích theo Điểm tín dụng")
plt.figure(figsize=(10, 6))
sns.countplot(x='credit_score_group', hue='default_status', data=df, palette='plasma', order=df['credit_score_group'].cat.categories)
plt.title('Phân phối Nhóm Điểm tín dụng theo Trạng thái Vỡ nợ')
plt.xlabel('Nhóm Điểm tín dụng')
plt.ylabel('Số lượng khách hàng')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Trạng thái Vỡ nợ')
plt.tight_layout()
plt.show()

# Phân tích Khoản tiết kiệm
print("\n#### Phân tích theo Khoản tiết kiệm")
plt.figure(figsize=(10, 6))
sns.histplot(df['savings_amount_vnd'], bins=30, kde=True, color='skyblue') # color argument is fine
plt.title('Phân phối Khoản tiết kiệm')
plt.xlabel('Khoản tiết kiệm (VND)')
plt.ylabel('Tần suất')
plt.ticklabel_format(style='plain', axis='x') # Avoid scientific notation on x-axis
plt.show()

plt.figure(figsize=(10, 6))
sns.boxplot(x='default_status', y='savings_amount_vnd', data=df, palette='rocket')
plt.title('Khoản tiết kiệm theo Trạng thái Vỡ nợ')
plt.xlabel('Trạng thái Vỡ nợ')
plt.ylabel('Khoản tiết kiệm (VND)')
plt.ticklabel_format(style='plain', axis='y') # Avoid scientific notation on y-axis
plt.show()

# Ghi chú: Thêm đoạn Markdown với kết luận sau các biểu đồ này.
# Ví dụ:
# '''
# **Kết luận 3.3.2 (Khoản tiết kiệm):** Biểu đồ phân phối cho thấy phần lớn khách hàng có khoản tiết kiệm thấp.
# Biểu đồ boxplot có thể chỉ ra rằng những khách hàng vỡ nợ có khoản tiết kiệm trung bình thấp hơn hoặc phân phối khác biệt so với nhóm thanh toán đầy đủ.
# '''

# Phân tích Mục đích vay
print("\n#### Phân tích theo Mục đích vay")
plt.figure(figsize=(12, 7))
sns.countplot(x='loan_purpose', hue='default_status', data=df, palette='coolwarm')
plt.title('Phân phối Mục đích vay theo Trạng thái Vỡ nợ')
plt.xlabel('Mục đích vay')
plt.ylabel('Số lượng khách hàng')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Trạng thái Vỡ nợ')
plt.tight_layout()
plt.show()

# 3.4. Ma trận tương quan (Correlation Matrix) giữa các biến số nguyên
print("\n### 3.4. Ma trận tương quan giữa các biến số nguyên")

# Chọn các cột số nguyên hoặc số thực
numerical_cols = df.select_dtypes(include=np.number).columns

# Cần đảm bảo 'is_defaulted' (đã được tạo từ 'default_status') là dạng số để đưa vào ma trận tương quan.
# Các cột số tiền có thể có mối tương quan mạnh.
cols_for_corr = ['age', 'years_at_current_job', 'credit_score',
                 'existing_debt_vnd', 'savings_amount_vnd', 'requested_loan_amount_vnd',
                 'loan_term_months', 'is_defaulted']

# Đảm bảo chỉ chọn các cột có trong DataFrame
actual_cols_for_corr = [col for col in cols_for_corr if col in df.columns]

if len(actual_cols_for_corr) > 1:
    correlation_matrix = df[actual_cols_for_corr].corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
    plt.title('Ma trận Tương quan giữa các biến số nguyên')
    plt.show()
else:
    print("Không đủ cột số để tạo ma trận tương quan.")

Dựa trên phân tích EDA đã thực hiện, đây là một số nhận xét chính:

1.  **Tỷ lệ nợ xấu chung**: Khoảng 31.01% tổng số khách hàng đã bị nợ xấu (Defaulted).
2.  **Mối liên hệ giữa thu nhập và nợ xấu**: Có một mối tương quan rõ rệt giữa mức thu nhập và tỷ lệ nợ xấu. Các nhóm thu nhập thấp có tỷ lệ nợ xấu cao hơn đáng kể (ví dụ: nhóm 'Thấp (Low)' có tỷ lệ nợ xấu khoảng 42.32%), trong khi các nhóm thu nhập cao hơn có tỷ lệ nợ xấu thấp hơn nhiều (ví dụ: nhóm 'Rất cao (Very High)' chỉ khoảng 14.87%). Điều này cho thấy thu nhập là một yếu tố dự báo mạnh mẽ về khả năng trả nợ.
3.  **Mục đích vay và nợ xấu**: Tỷ lệ nợ xấu không có sự khác biệt quá lớn giữa các mục đích vay. Các mục đích như 'Học vấn' và 'Mua xe' có tỷ lệ nợ xấu nhỉnh hơn một chút, trong khi 'Y tế' có tỷ lệ thấp hơn.
4.  **Độ tuổi và nợ xấu**: Khách hàng trẻ tuổi (20-30) có tỷ lệ nợ xấu cao hơn một chút (khoảng 32.77%) so với nhóm trung niên và lớn tuổi hơn (khoảng 30.5%).

Các biểu đồ phân phối biến số cho thấy dữ liệu tương đối cân bằng và không có các giá trị thiếu (null) đáng kể, giúp đảm bảo chất lượng cho các phân tích tiếp theo.

### **EDA Nâng cao: Phân tích Sức khỏe Tài chính và Rủi ro Vỡ nợ**

Phần này sẽ đi sâu vào các tỷ lệ tài chính quan trọng để hiểu rõ hơn sự khác biệt giữa khách hàng trả nợ đúng hạn và khách hàng vỡ nợ.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập phong cách cho biểu đồ
sns.set(style="whitegrid")

print("--- PHÂN TÍCH KHÁM PHÁ (EDA) ---\n")

# 1. Tỷ lệ nợ xấu (default_status)
default_counts = df['default_status'].value_counts(normalize=True) * 100
print("1. Tỷ lệ nợ xấu:")
print(default_counts)

plt.figure(figsize=(6, 6))
plt.pie(default_counts, labels=default_counts.index, autopct='%1.1f%%', colors=['#66b3ff','#ff9999'])
plt.title('Tỷ lệ khách hàng Defaulted vs Paid in full')
plt.show()

# 2. Mối liên hệ giữa thu nhập và nợ xấu
print("\n2. Mối liên hệ giữa thu nhập và nợ xấu:")
income_default = pd.crosstab(df['income_level_group'], df['default_status'], normalize='index') * 100
print(income_default)

income_default.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#ff9999', '#66b3ff'])
plt.title('Tỷ lệ nợ xấu theo Nhóm thu nhập')
plt.ylabel('Phần trăm (%)')
plt.xlabel('Nhóm thu nhập')
plt.xticks(rotation=45)
plt.legend(title='Trạng thái', loc='upper right')
plt.show()

# 3. Mối liên hệ giữa mục đích vay (loan_purpose) và nợ xấu
print("\n3. Mối liên hệ giữa mục đích vay và nợ xấu:")
purpose_default = pd.crosstab(df['loan_purpose'], df['default_status'], normalize='index') * 100
purpose_default = purpose_default.sort_values(by='Defaulted ', ascending=False)
print(purpose_default)

plt.figure(figsize=(12, 6))
sns.barplot(x=purpose_default.index, y=purpose_default['Defaulted '], palette='Reds_r')
plt.title('Tỷ lệ nợ xấu (%) theo Mục đích vay')
plt.ylabel('Tỷ lệ nợ xấu (%)')
plt.xlabel('Mục đích vay')
plt.xticks(rotation=45)
plt.show()

# 4. Độ tuổi và xu hướng trả nợ
print("\n4. Phân tích theo Độ tuổi:")
# Chia nhóm tuổi
bins_age = [20, 30, 45, 65]
labels_age = ['Trẻ (20-30)', 'Trung niên (31-45)', 'Lớn tuổi (46-65)']
df['age_group'] = pd.cut(df['age'], bins=bins_age, labels=labels_age)

age_default = pd.crosstab(df['age_group'], df['default_status'], normalize='index') * 100
print(age_default)

age_default.plot(kind='bar', figsize=(10, 6), color=['#ff9999', '#66b3ff'])
plt.title('Tỷ lệ nợ xấu theo Nhóm tuổi')
plt.ylabel('Phần trăm (%)')
plt.xlabel('Nhóm tuổi')
plt.xticks(rotation=0)
plt.show()

**1. Tỷ lệ nợ xấu:** Tổng cộng có 31.0% khách hàng rơi vào trạng thái 'Defaulted' (vỡ nợ) và 69.0% khách hàng đã thanh toán đầy đủ ('Paid in full').

**2. Thu nhập và nợ xấu:** Có mối liên hệ rất rõ rệt. Nhóm Thu nhập thấp (Low) có tỷ lệ vỡ nợ cao nhất (42.3%). Tỷ lệ này giảm dần khi thu nhập tăng lên, chỉ còn 14.9% ở nhóm Rất cao (Very High). Như vậy, người thu nhập thấp có xu hướng dễ vỡ nợ hơn.

**3. Mục đích vay:** Rủi ro giữa các mục đích vay khá tương đồng, tuy nhiên vay cho Học vấn (31.7%) và Mua xe (31.5%) có tỷ lệ vỡ nợ nhỉnh hơn một chút so với các mục đích khác như Y tế hay Hợp nhất nợ.

**4. Độ tuổi:** Nhóm khách hàng Trẻ (20-30) có tỷ lệ vỡ nợ cao nhất (32.8%). Khách hàng Lớn tuổi (46-65) có xu hướng trả nợ tốt nhất với tỷ lệ vỡ nợ thấp nhất (30.4%).

In [ ]:
print("\n--- Phân tích Tỷ lệ Tài chính theo Trạng thái Nợ xấu ---")
financial_ratios = [
    'debt_to_income_ratio',
    'savings_to_income_ratio',
    'loan_amount_to_income_ratio',
    'net_financial_position'
]

plt.figure(figsize=(16, 12))
for i, col in enumerate(financial_ratios):
    plt.subplot(2, 2, i + 1)
    sns.violinplot(x='default_status', y=col, data=df, palette='coolwarm')
    plt.title(f'Phân phối {col} theo Trạng thái Nợ xấu')
    plt.xlabel('Trạng thái Nợ xấu')
    plt.ylabel(col)
    # Limit y-axis for better visualization of main distribution, if values are extremely skewed
    if col != 'net_financial_position': # net_financial_position can have large negative values
        plt.ylim(df[col].quantile(0.01), df[col].quantile(0.99)) # show 98% of data

plt.tight_layout()
plt.show()

# **Insight từ Hành vi và Nhân khẩu học:**

* **Về mức thu nhập:** Nhóm khách hàng có thu nhập Trung bình (8 - 20 triệu VNĐ) chiếm tỷ trọng lớn nhất trong tệp dữ liệu. Đáng chú ý, tỷ lệ vỡ nợ (Default) tập trung dày đặc ở nhóm thu nhập Thấp (< 8 triệu VNĐ) và nhóm Thất nghiệp/Làm việc bán thời gian.

* **Về điểm tín dụng & Mục đích vay:** Khách hàng có điểm tín dụng "Trung bình" và "Tốt" có nhu cầu vay vốn cao nhất, chủ yếu phục vụ cho các mục đích như Y tế, Sửa nhà và Mua sắm thiết bị. Nhóm có thời gian làm việc hiện tại (Years at current job) càng lâu, tỷ lệ trả nợ đúng hạn càng cao, chứng tỏ sự ổn định trong dòng tiền cá nhân.

# **4. Kỹ thuật Đặc trưng (Feature Engineering)**

### **A. Nhóm biến Chỉ số Tài chính cốt lõi (Financial Ratios):**

In [ ]:

print("### 4.1. Các chỉ số thống kê tổng quan")

# Tổng hồ sơ vay
total_applications = len(df_sheet2)
print(f"Tổng số hồ sơ vay: {total_applications}")

# Tỷ lệ vỡ nợ
default_rate = df_sheet2['is_defaulted'].mean() * 100
print(f"Tỷ lệ vỡ nợ: {default_rate:.2f}%")

# Tổng khoản vay được yêu cầu
total_loan_requested = df_sheet2['requested_loan_amount_vnd'].sum()
print(f"Tổng số tiền vay được yêu cầu: {total_loan_requested:,.0f} VND")

# 4.2. Tạo cột Debt-to-Savings Ratio (Tỷ lệ Nợ/Tiết kiệm)
# Tránh lỗi chia cho 0 bằng cách thay thế 0 trong savings_amount_vnd bằng NaN, sau đó thay thế NaN bằng 0 sau khi chia.
df_sheet2['debt_to_savings_ratio'] = df_sheet2['existing_debt_vnd'] / df_sheet2['savings_amount_vnd'].replace(0, np.nan)
# Thay thế inf (division by zero) và NaN (savings_amount_vnd was 0) bằng 0 hoặc một giá trị phù hợp
df_sheet2['debt_to_savings_ratio'] = df_sheet2['debt_to_savings_ratio'].replace([np.inf, -np.inf], np.nan)
df_sheet2['debt_to_savings_ratio'] = df_sheet2['debt_to_savings_ratio'].fillna(0) # Hoặc giá trị trung bình/trung vị nếu muốn

print("\nĐã tạo cột 'debt_to_savings_ratio'. Vài dòng đầu:")
display(df_sheet2[['existing_debt_vnd', 'savings_amount_vnd', 'debt_to_savings_ratio']].head())

# 4.3. Tạo cột Loan-to-Income Ratio (Tỷ lệ khoản vay/Thu nhập ước tính)
# Đầu tiên, tạo cột thu nhập ước tính dạng số từ 'income_level'
income_estimate_map = {
    'Low': 4000000,       # Dưới 8 triệu VND
    'Medium': 14000000,   # (8 + 20) / 2 = 14 triệu VND
    'High': 32500000,     # (20 + 45) / 2 = 32.5 triệu VND
    'Very High': 50000000 # Ước tính trên 45 triệu VND
}

df_sheet2['estimated_monthly_income_vnd'] = df_sheet2['income_level'].map(income_estimate_map)

# Tính Loan-to-Income Ratio
# Tránh lỗi chia cho 0
df_sheet2['loan_to_income_ratio'] = df_sheet2['requested_loan_amount_vnd'] / df_sheet2['estimated_monthly_income_vnd'].replace(0, np.nan)
df_sheet2['loan_to_income_ratio'] = df_sheet2['loan_to_income_ratio'].replace([np.inf, -np.inf], np.nan)
df_sheet2['loan_to_income_ratio'] = df_sheet2['loan_to_income_ratio'].fillna(0) # Thay thế inf và NaN bằng 0

print("\nĐã tạo cột 'loan_to_income_ratio'. Vài dòng đầu:")
display(df_sheet2[['requested_loan_amount_vnd', 'estimated_monthly_income_vnd', 'loan_to_income_ratio']].head())

# **5. Xây dựng Mô hình Học máy (Machine Learning Modeling)**

In [ ]:
#Đảm bảo các cột đặc trưng tài chính tồn tại
income_mapping = {
    'Dưới 8 triệu VND/tháng': 6000000,
    'Từ 8 - 20 triệu VND/tháng': 14000000,
    'Từ 20 - 45 triệu VND/tháng': 32500000,
    'Trên 45 triệu VND/tháng': 60000000
}
df['income_level_numeric'] = df['income_level'].map(income_mapping)

df['debt_to_income_ratio'] = df['existing_debt_vnd'] / df['income_level_numeric']
df['savings_to_income_ratio'] = df['savings_amount_vnd'] / df['income_level_numeric']
df['loan_amount_to_income_ratio'] = df['requested_loan_amount_vnd'] / df['income_level_numeric']
df['net_financial_position'] = df['savings_amount_vnd'] - df['existing_debt_vnd']

# Định nghĩa lại danh sách đặc trưng
features = [
    'age',
    'years_at_current_job',
    'credit_score',
    'existing_debt_vnd',
    'savings_amount_vnd',
    'requested_loan_amount_vnd',
    'loan_term_months',
    'debt_to_income_ratio',
    'savings_to_income_ratio',
    'loan_amount_to_income_ratio',
    'net_financial_position'
]

# Xử lý các biến phân loại và định nghĩa biến mục tiêu
le = LabelEncoder()
categorical_cols = ['income_level', 'employment_status', 'home_ownership_status', 'loan_purpose']

for col in categorical_cols:
    df[col + '_encoded'] = le.fit_transform(df[col])

# Mã hóa biến mục tiêu 'default_status' (xử lý khoảng trắng nếu có)
df['default_status_clean'] = df['default_status'].str.strip()
df['default_status_encoded'] = le.fit_transform(df['default_status_clean'])

#  Định nghĩa các đặc trưng (X) và biến mục tiêu (y)
X_cols = features + [col + '_encoded' for col in categorical_cols]
y = df['default_status_encoded']
X = df[X_cols].fillna(0) # Xử lý giá trị NaN nếu có

# Chuẩn hóa dữ liệu X
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Chia dữ liệu thành tập huấn luyện và kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

#  Xây dựng mô hình Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

#  Đánh giá mô hình
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nROC AUC Score:", roc_auc_score(y_test, y_pred_proba))

# Trực quan hóa ma trận nhầm lẫn
plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Paid in Full', 'Defaulted'], yticklabels=['Paid in Full', 'Defaulted'])
plt.xlabel('Dự đoán')
plt.ylabel('Thực tế')
plt.title('Ma trận nhầm lẫn')
plt.show()

Dự án sử dụng thuật toán Random Forest Classifier. Đây là mô hình dạng tập hợp (Ensemble Learning) hoạt động cực kỳ hiệu quả với dữ liệu dạng bảng (Tabular Data), giúp nắm bắt tốt các mối quan hệ phi tuyến tính giữa thu nhập, tiền tiết kiệm và rủi ro tín dụng mà không dễ bị quá khớp (overfitting) như các mô hình cây quyết định đơn lẻ.

# **6. Phân khúc Thị trường (Market Segmentation)**

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Chuẩn bị dữ liệu cho phân cụm
# Tạo lại cột income_level_numeric để tránh lỗi KeyError
income_mapping = {
    'Dưới 8 triệu VND/tháng': 6000000,
    'Từ 8 - 20 triệu VND/tháng': 14000000,
    'Từ 20 - 45 triệu VND/tháng': 32500000,
    'Trên 45 triệu VND/tháng': 60000000
}
df['income_level_numeric'] = df['income_level'].map(income_mapping)

# Chọn các biến quan trọng
clustering_features = ['age', 'income_level_numeric', 'credit_score', 'existing_debt_vnd']
X_cluster = df[clustering_features].copy()

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# 2. Áp dụng K-Means (K=4)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster_persona'] = kmeans.fit_predict(X_scaled)

# 3. Phân tích kết quả
default_map = {'Paid in full': 0, 'Defaulted ': 1}
df['default_numeric'] = df['default_status'].map(default_map)

cluster_summary = df.groupby('cluster_persona').agg({
    'age': 'mean',
    'income_level_numeric': 'mean',
    'credit_score': 'mean',
    'existing_debt_vnd': 'mean',
    'default_numeric': 'mean',
    'customer_id': 'count'
}).rename(columns={'default_numeric': 'default_rate', 'customer_id': 'count'})

print("=== PHÂN KHÚC KHÁCH HÀNG CHUYÊN SÂU (CLUSTERING) ===")
print(cluster_summary)

# 4. Trực quan hóa
plt.figure(figsize=(10, 6))
sns.barplot(x=cluster_summary.index, y=cluster_summary['default_rate'] * 100, palette='viridis')
plt.title('Tỷ lệ nợ xấu (%) theo từng Cụm khách hàng')
plt.ylabel('Tỷ lệ nợ xấu (%)')
plt.xlabel('Cụm (Persona)')
plt.show()

print("\nNhận xét về các cụm:")
for i in range(4):
    c = cluster_summary.loc[i]
    print(f"- Cụm {i}: Tuổi TB {c['age']:.1f}, Thu nhập TB {c['income_level_numeric']/1e6:.1f}M, Điểm tín dụng TB {c['credit_score']:.0f}. Tỷ lệ nợ xấu: {c['default_rate']*100:.1f}%")

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Sử dụng các đặc trưng số đã chuẩn hóa
wcss = []
K_range = range(1, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, wcss, 'bx-')
plt.xlabel('Số lượng cụm (k)')
plt.ylabel('WCSS (Inertia)')
plt.title('Phương pháp Elbow xác định K tối ưu')
plt.show()

Kết hợp thuật toán phân cụm K-Means và lý thuyết Ma trận Tăng trưởng (Growth-Share Matrix), tệp khách hàng được chia thành 3 nhóm chiến lược:

**Phân khúc Ngôi sao (Star):** Nhóm Thu nhập Cao/Rất cao + Điểm tín dụng Xuất sắc. Rủi ro gần như bằng 0, tuy nhiên quy mô tệp khách hàng này nhỏ, khó mở rộng thị phần (Scale-up) đột phá.

**Phân khúc Bò sữa (Cash Cow):** Nhóm Thu nhập Trung bình/Cao (8 - 45 triệu VNĐ) + Điểm tín dụng Tốt. Đây là nhóm có quy mô khổng lồ, nhu cầu vay thường xuyên, rủi ro vỡ nợ ở mức kiểm soát được.

**Phân khúc Chó mực (Dog):** Nhóm Thu nhập Thấp + Thất nghiệp. Nhu cầu vay cao nhưng tỷ lệ nợ xấu vượt mức an toàn, sinh lời âm do chi phí thu hồi nợ cao.

In [ ]:
from sklearn.metrics import classification_report, roc_curve, auc, precision_recall_curve

# 1. Chi tiết Precision, Recall, F1
print("Báo cáo chi tiết hiệu suất mô hình:")
print(classification_report(y_test, y_pred))

# 2. Vẽ đường cong ROC
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, color='darkorange', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend()

# 3. Vẽ đường cong Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
plt.subplot(1, 2, 2)
plt.plot(recall, precision, color='blue')
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.tight_layout()
plt.show()

# **7. Giải thích Mô hình & Định lượng Rủi ro (Model Explainability with SHAP)**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

print("--- PHÂN TÍCH NHÂN QUẢ & MÔ HÌNH HÓA RỦI RO ---")

# 1. Khôi phục/Tạo lại các đặc trưng cần thiết
income_mapping = {
    'Dưới 8 triệu VND/tháng': 6000000,
    'Từ 8 - 20 triệu VND/tháng': 14000000,
    'Từ 20 - 45 triệu VND/tháng': 32500000,
    'Trên 45 triệu VND/tháng': 60000000
}
df['income_level_numeric'] = df['income_level'].map(income_mapping)
df['debt_to_income_ratio'] = df['existing_debt_vnd'] / df['income_level_numeric']
df['savings_to_income_ratio'] = df['savings_amount_vnd'] / df['income_level_numeric']
df['loan_amount_to_income_ratio'] = df['requested_loan_amount_vnd'] / df['income_level_numeric']
df['net_financial_position'] = df['savings_amount_vnd'] - df['existing_debt_vnd']

le = LabelEncoder()
for col in ['income_level', 'employment_status', 'home_ownership_status', 'loan_purpose']:
    df[f"{col}_encoded"] = le.fit_transform(df[col])

df['default_status_encoded'] = le.fit_transform(df['default_status'])

cat_encoded_cols = ['income_level_encoded', 'employment_status_encoded', 'home_ownership_status_encoded', 'loan_purpose_encoded']
num_cols_for_model = ['age', 'years_at_current_job', 'credit_score', 'existing_debt_vnd', 'savings_amount_vnd', 'requested_loan_amount_vnd', 'loan_term_months', 'debt_to_income_ratio', 'savings_to_income_ratio', 'loan_amount_to_income_ratio', 'net_financial_position']
X_cols = num_cols_for_model + cat_encoded_cols

X = df[X_cols]
y = df['default_status_encoded']

# 2. Huấn luyện lại mô hình để đảm bảo biến 'model' tồn tại
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X, y)

# 3. Heatmap Tương quan
plt.figure(figsize=(12, 8))
correlation_data = X.copy()
correlation_data['default_status_target'] = y
corr_matrix = correlation_data.corr()
sns.heatmap(corr_matrix[['default_status_target']].sort_values(by='default_status_target', ascending=False), annot=True, cmap='coolwarm', center=0)
plt.title('Mối tương quan giữa các yếu tố và Trạng thái Vỡ nợ')
plt.show()

# 4. Feature Importance
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({'Yếu tố': X_cols, 'Độ quan trọng': importances}).sort_values(by='Độ quan trọng', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Độ quan trọng', y='Yếu tố', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance - Các yếu tố quyết định rủi ro vỡ nợ')
plt.show()

print("Top 5 yếu tố quan trọng nhất ảnh hưởng đến nợ xấu:")
print(feature_importance_df.head(5))

Kết quả phân tích đã chỉ ra những Insight cực kỳ quan trọng cho chiến lược quản trị rủi ro:

**1. Yếu tố then chốt:** credit_score (điểm tín dụng) là biến số quan trọng nhất với trọng số vượt trội (~43%). Điều này khẳng định hệ thống xếp hạng tín dụng hiện tại là 'xương sống' trong việc dự báo.

**2. Tình trạng tài chính:** Các yếu tố như số dư tiết kiệm (savings_amount_vnd), nợ hiện hữu (existing_debt_vnd) và vị thế tài chính ròng (net_financial_position) nằm trong Top 5. Điều này cho thấy khả năng thanh khoản và tích lũy là bộ đệm quan trọng giúp khách hàng tránh vỡ nợ.

**3. Tương quan:** Heatmap cho thấy mối tương quan thuận giữa tỷ lệ nợ/thu nhập (debt_to_income_ratio) và rủi ro, trong khi điểm tín dụng có tương quan nghịch mạnh mẽ.

In [ ]:
import shap
import numpy as np

# 1. Khởi tạo SHAP Explainer
explainer = shap.TreeExplainer(model)

# 2. Tính toán SHAP values
# Lấy mẫu 500 dòng từ tập test đã chuẩn hóa
X_sample = X_test.iloc[:500]
shap_values = explainer.shap_values(X_sample)

# Kiểm tra cấu trúc shap_values để tránh lỗi AssertionError
# Đối với binary classification trong Random Forest, shap_values có thể là list [n_samples, n_features, 2]
# hoặc [2, n_samples, n_features]. Ta cần lấy index 1 (Defaulted).
if isinstance(shap_values, list):
    # Nếu là list, lấy mảng ứng với class 1
    shap_to_plot = shap_values[1]
elif len(shap_values.shape) == 3:
    # Nếu là mảng 3 chiều, lấy slice cuối cùng
    shap_to_plot = shap_values[:, :, 1]
else:
    # Trường hợp còn lại
    shap_to_plot = shap_values

# 3. Trực quan hóa Summary Plot (Bar)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_to_plot, X_sample, plot_type="bar", show=False)
plt.title("SHAP Feature Importance (Lớp: Defaulted)")
plt.show()

# 4. Summary Plot (Beeswarm)
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_to_plot, X_sample, show=False)
plt.title("Tác động của các yếu tố đến xác suất nợ xấu")
plt.show()

**Insight từ SHAP:**
* **Credit Score:** Có tác động mạnh nhất. Giá trị Credit Score thấp (màu xanh) đẩy xác suất nợ xấu lên cao (SHAP value dương).
* **Savings Amount:** Các khách hàng có tiết kiệm thấp có xu hướng bị mô hình đánh giá rủi ro cao hơn.
* **Net Financial Position:** Khẳng định lại giả thuyết của chúng ta về 'Hidden Gems' - vị thế tài chính ròng dương giúp kéo giảm xác suất nợ xấu một cách đáng kể.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Chuẩn bị dữ liệu so sánh
avg_total = df[['income_level_numeric', 'credit_score', 'savings_amount_vnd', 'existing_debt_vnd']].mean()
avg_gems = untapped_segment[['income_level_numeric', 'credit_score', 'savings_amount_vnd', 'existing_debt_vnd']].mean()

comparison_df = pd.DataFrame({
    'Chỉ số': ['Thu nhập trung bình', 'Điểm tín dụng', 'Tiền tiết kiệm', 'Nợ hiện tại'],
    'Tổng thể KH': [avg_total['income_level_numeric'], avg_total['credit_score'], avg_total['savings_amount_vnd'], avg_total['existing_debt_vnd']],
    'Hidden Gems': [avg_gems['income_level_numeric'], avg_gems['credit_score'], avg_gems['savings_amount_vnd'], avg_gems['existing_debt_vnd']]
})

# 2. Vẽ Dashboard chân dung
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Biểu đồ cột so sánh Thu nhập và Tài sản (Log scale để dễ nhìn)
metrics_to_plot = ['Thu nhập trung bình', 'Tiền tiết kiệm', 'Nợ hiện tại']
plot_data = comparison_df[comparison_df['Chỉ số'].isin(metrics_to_plot)].melt(id_vars='Chỉ số', var_name='Nhóm', value_name='Giá trị (VND)')

sns.barplot(data=plot_data, x='Chỉ số', y='Giá trị (VND)', hue='Nhóm', ax=axes[0], palette=['#A0A0A0', '#2ECC71'])
axes[0].set_yscale('log')
axes[0].set_title('So sánh Tài chính: Hidden Gems vs. Tổng thể (Log scale)')
axes[0].set_ylabel('Giá trị VND (Log scale)')

# Biểu đồ so sánh Điểm tín dụng (Vốn là điểm yếu của Gems)
score_data = comparison_df[comparison_df['Chỉ số'] == 'Điểm tín dụng'].melt(id_vars='Chỉ số', var_name='Nhóm', value_name='Điểm')
sns.barplot(data=score_data, x='Nhóm', y='Điểm', ax=axes[1], palette=['#A0A0A0', '#E74C3C'])
axes[1].set_ylim(300, 850)
axes[1].set_title('Nghịch lý: Điểm tín dụng của Hidden Gems thấp hơn')
axes[1].axhline(670, color='blue', linestyle='--', label='Ngưỡng tín dụng Tốt')
axes[1].legend()

plt.suptitle('DASHBOARD CHÂN DUNG KHÁCH HÀNG HIDDEN GEMS', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print(f"--- TÓM TẮT CHÂN DUNG ---")
print(f"1. Thu nhập: Cao hơn trung bình {(avg_gems['income_level_numeric']/avg_total['income_level_numeric']):.1f} lần.")
print(f"2. Tiết kiệm: Cao hơn trung bình {(avg_gems['savings_amount_vnd']/avg_total['savings_amount_vnd']):.1f} lần.")
print(f"3. Nợ: Thấp hơn trung bình {(avg_total['existing_debt_vnd']/avg_gems['existing_debt_vnd']):.1f} lần.")
print(f"4. Điểm tín dụng: Thấp hơn {(avg_total['credit_score'] - avg_gems['credit_score']):.0f} điểm so với mặt bằng chung.")

**Nghịch lý tài chính:** Mặc dù điểm tín dụng của nhóm này thấp hơn trung bình 42 điểm, nhưng thu nhập của họ cao gấp 2.2 lần và lượng tiền tiết kiệm cao gấp 3.9 lần so với mặt bằng chung.

**Rủi ro thực tế thấp:** Tỷ lệ nợ của họ chỉ bằng 0.5 lần so với khách hàng thông thường. Điều này chứng minh rằng họ không phải là đối tượng rủi ro, mà chỉ đơn giản là những người chưa được hệ thống cũ hiểu đúng.

# **8. Đề xuất Chiến lược Kinh doanh (Business Recommendations)**

In [ ]:
# 1. TỔNG HỢP DỮ LIỆU CHIẾN LƯỢC (BÁO CÁO CỤM & MỤC ĐÍCH VAY)
# Tạo bảng pivot để xem rủi ro theo mục đích vay trong từng cụm
strategy_report = df.groupby(['cluster_persona', 'loan_purpose']).agg({
    'customer_id': 'count',
    'default_numeric': 'mean',
    'credit_score': 'mean',
    'income_level_numeric': 'mean'
}).rename(columns={'default_numeric': 'Tỷ lệ nợ xấu', 'customer_id': 'Số lượng KH'}).reset_index()

# 2. TRỰC QUAN HÓA (MÔ PHỎNG DASHBOARD)
fig, ax = plt.subplots(1, 2, figsize=(20, 8))

# Biểu đồ 1: Ma trận Rủi ro theo Cụm và Mục đích vay
pivot_default = strategy_report.pivot(index="cluster_persona", columns="loan_purpose", values="Tỷ lệ nợ xấu")
sns.heatmap(pivot_default * 100, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax[0])
ax[0].set_title("Dashboard: Tỷ lệ Nợ xấu (%) theo Cụm và Mục đích vay")
ax[0].set_ylabel("Nhóm Khách hàng (Cluster Persona)")
ax[0].set_xlabel("Mục đích vay")

# Biểu đồ 2: Quy mô khách hàng và rủi ro trung bình
cluster_summary_plot = strategy_report.groupby('cluster_persona').agg({'Số lượng KH': 'sum', 'Tỷ lệ nợ xấu': 'mean'}).reset_index()
sns.barplot(x='cluster_persona', y='Số lượng KH', data=cluster_summary_plot, palette='viridis', ax=ax[1])
ax2 = ax[1].twinx()
sns.lineplot(x='cluster_persona', y='Tỷ lệ nợ xấu', data=cluster_summary_plot, color='red', marker='o', ax=ax2)
ax[1].set_title("Quy mô Khách hàng & Tỷ lệ Nợ xấu trung bình mỗi cụm")

plt.tight_layout()
plt.show()

# 3. ĐỀ XUẤT CHÍNH SÁCH TÍN DỤNG (STRATEGY RECOMMENDATIONS)
print("=== ĐỀ XUẤT CHÍNH SÁCH TÍN DỤNG THEO PHÂN KHÚC (CREDIT POLICY) ===")

# Dựa trên kết quả clustering:
# Cluster 2 (VIPs): An toàn nhất
# Cluster 0: Trung bình - Khá
# Cluster 1 & 3: Rủi ro cao

policies = {
    "Nhóm AN TOÀN (Cụm 0 & 2)": 
        "- Ưu tiên: Giảm lãi suất vay từ 1-2% so với lãi suất sàn.\n- Hành động: Tăng hạn mức thẻ tín dụng tự động, phê duyệt siêu tốc cho vay tiêu dùng.",
    "Nhóm RỦI RO CAO (Cụm 1 & 3)": 
        "- Ưu tiên: Quản trị rủi ro.\n- Hành động: Yêu cầu tài sản đảm bảo (Collateral) hoặc người bảo lãnh. Thắt chặt tỷ lệ DTI < 30%.",
    "Nhóm TIỀM NĂNG (Thu nhập cao nhưng Credit Score thấp)": 
        "- Ưu tiên: Xây dựng lịch sử tín dụng.\n- Hành động: Cung cấp các gói vay ngắn hạn (3-6 tháng) với hạn mức nhỏ để khách hàng chứng minh khả năng chi trả."
}

for group, policy in policies.items():
    print(f"\n[{group}]\n{policy}")

Dựa trên các phân tích định lượng và định tính, nhóm đề xuất Ban quản lý Sản phẩm tập trung nguồn lực phát triển cho Phân khúc Bò sữa (Cash Cow) - Nhóm khách hàng thu nhập Trung bình đến Cao (8 - 45 triệu VNĐ) có công việc toàn thời gian.

Giải pháp triển khai cụ thể:

* **Sản phẩm linh hoạt:** Thiết kế các gói vay tín chấp có hạn mức được cá nhân hóa tự động hóa dựa trên điểm SHAP của mô hình Học máy.

* **Lãi suất động (Dynamic Pricing):** Khách hàng trong phân khúc Cash Cow nếu chứng minh được số dư tiết kiệm (savings_amount_vnd) cao sẽ được hệ thống Fintech tự động giảm lãi suất vay, nhằm tăng tính cạnh tranh với các ngân hàng truyền thống.

* **Quản trị rủi ro:** Áp dụng mô hình Random Forest như một bộ lọc (filter) thời gian thực trên App Fintech. Các hồ sơ bị AI đánh dấu rủi ro cao (thuộc nhóm Dog) sẽ bị từ chối tự động hoặc yêu cầu bổ sung tài sản thế chấp để giảm thiểu nợ xấu (NPL).